# Proyecto 2 - Introducción a la Inteligencia Artificial


## 🧾 Selección y justificación del dataset

Se utilizó la base de datos **“Wine Quality”** disponible en **Kaggle** (autor: *rajyellow46*, fuente abierta), la cual contiene información fisicoquímica de vinos tintos y blancos.  
Este conjunto de datos cuenta con **más de 6 000 registros**, por lo que cumple con el requisito de tamaño mínimo (≥ 5000 instancias).  
Incluye **11 características numéricas** (por ejemplo: acidez, pH, densidad, alcohol, sulfatos, etc.) y una **variable objetivo `quality`** originalmente en una escala de 0 a 10, que se transformó en **4 categorías** (*baja*, *media-baja*, *media-alta* y *alta*) para los experimentos de clasificación.

Esta base es adecuada para el proyecto porque:

- Permite analizar cómo las variables fisicoquímicas afectan la calidad del vino.  
- Contiene datos **numéricos y categóricos**, adecuados para aplicar las técnicas de **preprocesamiento, clasificación y agrupamiento** requeridas.  
- Al incluir datos de vinos tintos y blancos, posibilita **comparaciones entre subgrupos** y análisis más amplios del dominio.  
- Su formato estructurado y origen público facilita la **reproducibilidad** y el cumplimiento de los lineamientos de la práctica.

Para garantizar la individualidad del conjunto de datos, se generó un subconjunto único con **(5000 + último dígito de la cédula de Manuel Zuleta Arango  × 100)** registros, utilizando un **random state distinto de 42**.


In [34]:
import kagglehub
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler



# Download latest version
path = kagglehub.dataset_download("rajyellow46/wine-quality")

print("Path to dataset files:", path)

# Lista los archivos descargados
print(os.listdir(path))

# Cargar los datos en un DataFrame de pandas
data = pd.read_csv(os.path.join(path, "winequalityN.csv"), sep=',')

# Mostrar las primeras filas del DataFrame
data.head()

def agrupar_calidad(q):
    if q <= 4:
        return 'baja'
    elif q == 5:
        return 'media-baja'
    elif q == 6:
        return 'media-alta'
    else:
        return 'alta'

data['quality'] = data['quality'].apply(agrupar_calidad)

print(data['quality'].value_counts())
print(len(data))
data.head()


Path to dataset files: C:\Users\manue\.cache\kagglehub\datasets\rajyellow46\wine-quality\versions\1
['winequalityN.csv']
quality
media-alta    2836
media-baja    2138
alta          1277
baja           246
Name: count, dtype: int64
6497


,type,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,white,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,media-alta
1,white,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,media-alta
2,white,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,media-alta
3,white,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,media-alta
4,white,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,media-alta


In [35]:
# --- Creación del subconjunto único de 5300 registros ---

RANDOM_STATE = 69 

# Crear el subconjunto aleatorio de 5300 registros
subset_data = data.sample(n=5300, random_state=RANDOM_STATE).reset_index(drop=True)

# Mostrar confirmación
print("Tamaño del subconjunto:", len(subset_data))
print(subset_data['quality'].value_counts())

print("✅ Subconjunto guardado como 'wine_quality_subset_5300.csv'")


Tamaño del subconjunto: 5300
quality
media-alta    2328
media-baja    1741
alta          1028
baja           203
Name: count, dtype: int64
✅ Subconjunto guardado como 'wine_quality_subset_5300.csv'


# Preprocesamiento del dataset

### Creamos las diferentes versiones del dataset 

In [ ]:
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
import numpy as np
import pandas as pd
import os

# Asumiendo que ya tienes 'subset_data' como tu dataset principal
data_base = subset_data.copy()

# Convertimos la variable categórica 'type' si existe (vino blanco/tinto)
data_base = pd.get_dummies(data_base, columns=['type'], drop_first=True)

# Separamos variables predictoras y objetivo
X = data_base.drop(columns=['quality'])
y = data_base['quality']

# --- FUNCIONES DE APOYO ---

def remove_outliers(df, threshold=0.05):
    """Elimina el 5% de valores más extremos de cada variable numérica."""
    df_no_outliers = df.copy()
    for col in df_no_outliers.select_dtypes(include=[np.number]).columns:
        low = df_no_outliers[col].quantile(threshold/2)
        high = df_no_outliers[col].quantile(1 - threshold/2)
        df_no_outliers = df_no_outliers[(df_no_outliers[col] >= low) & (df_no_outliers[col] <= high)]
    return df_no_outliers

def scale_data(X):
    """Aplica StandardScaler al conjunto de características."""
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return pd.DataFrame(X_scaled, columns=X.columns)

def balance_data(X, y):
    """Balancea las clases usando sobremuestreo aleatorio."""
    ros = RandomOverSampler(random_state=7)
    X_res, y_res = ros.fit_resample(X, y)
    return X_res, y_res

# --- CREACIÓN DE LAS 8 VERSIONES DEL DATASET ---

datasets = {}

for i in range(1, 9):
    temp_X, temp_y = X.copy(), y.copy()
    
    # Outliers
    if i in [3, 4, 7, 8]:  # versiones con outliers
        temp_X = remove_outliers(temp_X, threshold=0.05)
        temp_y = temp_y.loc[temp_X.index]
    
    # Balanceo
    if i in [2, 4, 6, 8]:  # versiones balanceadas
        temp_X, temp_y = balance_data(temp_X, temp_y)
    
    # Escalado (ED)
    if i in [5, 6, 7, 8]:  # ED(SI)
        temp_X = scale_data(temp_X)
    
    # Guardar resultado
    version_name = f"dataset_v{i}.csv"
    datasets[i] = pd.concat([temp_X, temp_y], axis=1)
    datasets[i].to_csv(f"data_sets/{version_name}", index=False)
    print(f"✅ Versión {i} guardada -> {version_name} | Filas: {len(datasets[i])}")

print("\nTodas las versiones creadas exitosamente.")


✅ Versión 1 guardada -> dataset_v1.csv | Filas: 5300
✅ Versión 2 guardada -> dataset_v2.csv | Filas: 9312
✅ Versión 3 guardada -> dataset_v3.csv | Filas: 3184
✅ Versión 4 guardada -> dataset_v4.csv | Filas: 5716
✅ Versión 5 guardada -> dataset_v5.csv | Filas: 5300
✅ Versión 6 guardada -> dataset_v6.csv | Filas: 9312
✅ Versión 7 guardada -> dataset_v7.csv | Filas: 4473
✅ Versión 8 guardada -> dataset_v8.csv | Filas: 5716

Todas las versiones creadas exitosamente.
